## GraphRAG

In this notebook we'll cover the GraphRAG using Neo4J. In our example we'll be indexing a Wikipedia URL, which we will vectorize (as usual) as well as generate a GraphDB (knowledge graph). 

<center>
<image src="../images/graph_rag.png" width="500" height="200"/>
</center>

When the user asks a question, it will be directed towards BOTH the vector-store (bottom of image above) against which we will do a keyword search & semantic search AND the graph DB (a Neo4J Aura [online instance]) in our case. All the 3 searches will extract a separate contexts, which will be combined and then fed to the LLM along with the user's question. The LLM's response will then be grounded in this (additional) context.

### What is a Knowledge Graph
A knowledge graph is a **structured representation of information** capturing _entities_, their _attributes_ and _relationships_. It models complex data and highlights connections within a domain. Some key components of a knowledge graph are:
* **_Entities_**: the fundamental unit of a knowledge graph representing **real-world objects, concepts or things** (e.g. _"Albert Einstein"_, _"Theory of relativity"_, _"University"_)
* **_Attributes_**: properties or characteristics or additional information about an entity (e.g. _"Albert Einstein"_ entity could have attribute such as _"birthday"_ with a value _"March 14, 1879"_)
* **_Relationships_**: Connections between entities describing how they are related to each other (e.g. _"Albert Einstein"_ is _related_ to _"Theory of Relativity"_ by the relationship _"developed"_)
* **_Nodes &amp; Edges_**: in a graphical representation, entities are nodes, and relationships are edges connecting those nodes.

Refer to the following diagram for a pictoral representation of the above concepts

<center>
<image src="../images/knowledge_graph.png" width="500" height="150"/>
</center>

### What is the problem with _basic_ RAG?
Let's understand the problem with _basic_ RAG using an example:
> Suppose a company has a large collection of internal documents, which includes research papers, technical reports, emails and meeting notes
> The goal is to answer the question _"What are the recent advances in our AI Research department"_

In the case of _basic_ RAG, it could land up searching for documents with terms like **_recent advancements_** and **_AI Research department_** and would then be retrieving the _top K_ documents based on vector similarity. The _response_ would be all sentences that have maximum similarity with these terms.

The final output could be something like:
* **Document 1**: _Our team has recently developed an AI model for Natural Language processing_
* **Document 2**: _In the past quarter we made significant progress in AI-based image recognition_
* **Document 3**: _AI advancements are at a rapid pace_
_Basic_ RAG works on vector similarity search, which may miss out on critical bits of information as it searches based on vector similarity only.

### GraphRAG
Conversely, GrapRAG uses a _knowledge graph_ instead of a vector DB for information retrieval. It is able to leverage the entities, their attributes and relationships to get more relevant and wholesome information to the LLM from which the LLM can generate a response.




In [1]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain.document_loaders import WikipediaLoader
from langchain_community.graphs.neo4j_graph import Neo4jGraph
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# since we are using Gemini, we'll use Google embeddings
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# I seem to have permanently exhausted rate limt on Google embeddings on free tier,
# I don't want to enable billing, so am switching to Cohere embeddings
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import FAISS

In [2]:
load_dotenv()  # load all API keys & Neo4J Aura connection parameters

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [4]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
# No longer using Google Embeddings as I have apparently exhausted my free quota :(
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/text-embedding-004", task_type="retrieval_document"
# )
embeddings = CohereEmbeddings(model="embed-english-v3.0")
faiss_store = pathlib.Path(os.getcwd()) / "../faiss_index_wiki_elizabeth1"

In [3]:
#### Loading the Wikipedia page on (queen) Elizabeth 1
# connect to Wikipedia & download article on Elizabeth I
from langchain_community.document_loaders import WikipediaLoader

loader = WikipediaLoader(query="Elizabeth_I", load_max_docs=5)
raw_documents = loader.load()
print(f"Loaded {len(raw_documents)} documents")

ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))